In [ ]:
import pandas as pd

# -----------------------------------------------------------
# Importa módulo de expressões regulares (não utilizado diretamente neste script,
# mas pode ser útil em extensões futuras)
import re

# Importa função para remover acentuação de strings
from unidecode import unidecode

# Importa a classe Document para manipular arquivos .docx
from docx import Document

# Importa Path para manipulação de caminhos de arquivos
from pathlib import Path

# Importa função para converter arquivos .docx em .pdf
from docx2pdf import convert

In [ ]:
exce = ('Desempenho 2026.1.xlsx')
caminh = ('G:/Drives compartilhados/Qualidade/AVALIAÇÃO DE DESEMPENHO/CQEHAF - Avaliação/')
df = pd.read_excel(caminh+exce)
df = df.loc[0:214]

In [ ]:
# ===========================================================
# CONFIGURAÇÕES
# ===========================================================
# ATENÇÃO: o arquivo modelo deve ser convertido de .doc para .docx
# apenas uma vez, manualmente, antes de usar este script.


# ===========================================================
# FUNÇÕES DE SUBSTITUIÇÃO
# ===========================================================

# Função principal para substituir placeholders no formato "<VARIAVEL>"
def replace_in_paragraph(paragraph, mapping):
    """
    Substitui placeholders em um parágrafo do Word,
    mesmo quando o texto está quebrado em múltiplos 'runs'.
    """

    # Junta todo o texto do parágrafo (todos os runs)
    full_text = "".join(run.text for run in paragraph.runs)

    # Se nenhum placeholder do mapping estiver presente no texto, sai da função
    if not any(k in full_text for k in mapping):
        return

    # Substitui cada placeholder pelo respectivo valor
    for k, v in mapping.items():
        full_text = full_text.replace(k, v)

    # Limpa o conteúdo de todos os runs existentes
    for i in range(len(paragraph.runs) - 1, -1, -1):
        paragraph.runs[i].text = ""

    # Reescreve o texto completo em um único run
    if paragraph.runs:
        paragraph.runs[0].text = full_text
    else:
        paragraph.add_run(full_text)


# Função para realizar substituições dentro de tabelas do Word
def replace_in_table(table, mapping):

    # Percorre todas as linhas da tabela
    for row in table.rows:

        # Percorre todas as células da linha
        for cell in row.cells:

            # Percorre todos os parágrafos dentro da célula
            for p in cell.paragraphs:
                replace_in_paragraph(p, mapping)


# Função para substituir placeholders no cabeçalho e rodapé do documento
def replace_in_header_footer(doc, mapping):

    # Percorre todas as seções do documento (pode haver mais de uma)
    for section in doc.sections:

        # Processa o cabeçalho
        header = section.header
        if header is not None:

            # Substituição em parágrafos do cabeçalho
            for p in header.paragraphs:
                replace_in_paragraph(p, mapping)

            # Substituição em tabelas do cabeçalho
            for t in header.tables:
                replace_in_table(t, mapping)

        # Processa o rodapé
        footer = section.footer
        if footer is not None:

            # Substituição em parágrafos do rodapé
            for p in footer.paragraphs:
                replace_in_paragraph(p, mapping)

            # Substituição em tabelas do rodapé
            for t in footer.tables:
                replace_in_table(t, mapping)


# Função que preenche o documento Word a partir de um template
def preencher_documento(template_path, mapping, saida_path_docx):

    # Abre o documento modelo
    doc = Document(template_path)

    # Substituições no corpo principal do documento
    for paragraph in doc.paragraphs:
        replace_in_paragraph(paragraph, mapping)

    # Substituições dentro das tabelas do corpo
    for table in doc.tables:
        replace_in_table(table, mapping)

    # Substituições no cabeçalho e rodapé
    replace_in_header_footer(doc, mapping)

    # Salva o documento preenchido
    doc.save(saida_path_docx)

In [ ]:
nome_teste = df.loc[linha, 'Nomes']
crm_teste = df.loc[linha, 'CRM']
data_teste = df.loc[linha, 'data_nascimento']

In [ ]:
TEMPLATE = Path("arquivo_word.docx")
SAIDA_DIR = Path("relatorios_gerados")
SAIDA_DIR.mkdir(exist_ok=True)

mapping = {
    "<NOME>" : f'{nome_teste}',
    "<CRM>" : f"{crm_teste}", 
    "<DATA_DE_NASCIMENTO>": f"{data_teste}"
}

nome_base = f'{nome_teste}' + "_" + f'{crm_teste}'
saida_docx = SAIDA_DIR / f"{nome_base}.docx"

In [ ]:
# ===========================================================
# EXECUÇÃO FINAL
# ===========================================================

# Reimporta Path (redundante, mas mantido conforme código original)
from pathlib import Path

# Função principal para gerar relatório em DOCX e PDF
def criar_relatorio(TEMPLATE, mapping, saida_docx):

    # Preenche o documento Word a partir do template
    preencher_documento(TEMPLATE, mapping, saida_docx)

    # Define o caminho do arquivo PDF de saída
    saida_pdf = saida_docx.with_suffix('.pdf')

    # Converte o DOCX gerado em PDF
    convert(str(saida_docx), str(saida_pdf))

    # Opcional: remove o arquivo DOCX após conversão
    # saida_docx.unlink()

In [ ]:
def convert_date_format(date_str):
    date_str = date_str[3:5] + '/' + date_str[:2] + '/' + date_str[6:]
    return date_str

convert_date_format('24/06/2024')

In [ ]:
coluna = 'data_nascimento'
df['data_nascimento'] = pd.to_datetime(df['data_nascimento'], dayfirst=True, errors='coerce')
df['data_nascimento'] = df['data_nascimento'].dt.strftime('%d/%m/%Y')
print(df[coluna].unique())

In [ ]:
for linha in df.index:
    
    nome_texto = df.loc[linha, 'Nomes']
    data_texto = df.loc[linha, 'data_nascimento']
    crm_texto = df.loc[linha, 'CRM']
    
    print(nome_texto)
    
    mapping = {
    "<NOME>" : f'{nome_texto}',
    "<CRM>" : f"{crm_texto}", 
    "<DATA_DE_NASCIMENTO>": f"{data_texto}"
}
    nome_base = f'{nome_texto}' + "_" + f'{crm_texto}'
    saida_docx = SAIDA_DIR / f"{nome_base}.docx"
    
    criar_relatorio(TEMPLATE, mapping, saida_docx)
